# 🔎 **Coleta de Dados**

## Origem dos Dados



A DEV Community é uma comunidade com mais de três milhões de desenvolvedores de software [<a href="#ref1">1</a>]. No site, os desenvolvedores escrevem artigos (posts de blog, perguntas em fóruns de discussão, tópicos de ajuda, etc.), participam de discussões e constroem seus perfis profissionais [<a href="#ref2">2</a>].

O site é hospedado pelo FOREM [<a href="#ref2">2</a>], um software de código aberto para a construção de comunidades. Na documentação do FOREM é disponibilizada uma API pública [<a href="#ref3">3</a>] que retorna os textos publicados, tags sobre o assunto tratado, entre outras informações. A documentação da API não estabelece uma licença específica para os dados disponibilizados. Os dados que podem identificar de alguma forma os autores dos textos foram ocultados.

Para realizar a coleta dos dados foi utilizada a versão 1 [<a href="#ref4">4</a>] da API pública do DEV Community. 

#### ⚖️ Governança de Dados, Ética e Licenciamento




Os dados brutos coletados por este script são obtidos de forma pública, respeitando as diretrizes de taxa de requisição da plataforma (*rate-limiting* via pausas controladas no código). 

Em conformidade com as políticas do site DEV.to, os artigos textuais consumidos estão sob a licença padrão **Creative Commons Attribution-ShareAlike 4.0 International (CC BY-SA 4.0)**.

#### 🔄 Consumo e Retorno da API



⚠️ Descrição pendente
* incluir sobre os parametros de consulta da API 
* incluir sobre os dados de retorno> da API 

**Published articles (`/articles`)**

A documentação da API informa apenas o retorno 200 para esse endpoint, sem descrever nenhum código de erro para o caso de a tag passada como parâmetro não existir na base. Nesse cenário, a API retorna 200 com um array vazio (`[]`).

**Published article by id (`/articles/{id}`)**

Para esse endpoint, a documentação informa dois retornos possíveis: 200, para um id válido, e 404, para um id inexistente — que pode ocorrer, por exemplo, se um artigo for excluído entre o intervalo das duas chamadas (listagem e detalhe).

## A Coleta dos Dados e Tratamento de Erros

A proposta do projeto é simular um cenário real de mercado, onde os dados geralmente chegam com inconsistências e problemas de qualidade. Por isso, a coleta será feita de forma bruta, sem aplicar filtros ou validações neste primeiro momento, independentemente de os dados estarem corretos ou não. Com exceção do filtro dos artigos em inglês, pois o modelo será treinado nessa lingua.

Caso os dados se mostrem viáveis para o projeto, será realizado posteriormente um processo de ETL para tratamento, limpeza e padronização.

### Tratamento de erros e exceções

**Dados inexistentes ou vazios**

Considerando que os testes iniciais realizados no acesso à API não retornaram esse tipo de erro, não consideramos necessário um tratamento específico para os casos de dado inexistente ou vazio durante a coleta. Caso o retorno em grande quantidade nos mostre o contrário, esse tratamento será reavaliado. Caso a base seja utilizada para o MVP, esses casos serão tratados posteriormente, na etapa de limpeza dos dados.


**Erros de exceção**

Durante a coleta, serão tratados somente os erros de exceção (falhas na chamada em si, como timeout e/ou instabilidade do servidor).

## O Problema

Para realizar o projeto TechMind no Hackathon ONE, é necessário obter textos técnicos da área de tecnologia para treinar um modelo de classificação de textos. O modelo irá classificar os textos pelo assunto principal, que aqui estamos chamando de categoria. Os dados obtidos, através da API, não possuem um target que represente a categoria. Porém os artigos publicados no site Dev Community, possuem tags associadas a eles, que representam o tema do artigo. Conforme a documentação da API, essas tags podem ser utilizadas nos parametros da API para obter artigos especificos. Dito isso, se faz necessário realizar a Rotulagem de Dados (Data Labeling), gerando assim o target de cada texto. 

Além disso, será necessária a extração de uma amostra dos dados para validação manual dos textos, afim de conferir se os mesmos são realmente da categoria a que foi associado. Dessa forma iremos validar a qualidade dos dados e consequentemente viabilizar a utilização dos mesmos no projeto. 

## Objetivo



Este notebook tem como objetivo:

* realizar a coleta automatizada de artigos técnicos consumindo o endpoint público /api/articles da versão 1 [<a href="#ref4">4</a>]da API, usando como parâmetro as tags associadas a eles.
* filtrar somente os artigos em inglês.
* realizar a rotulação dos dados, associando um target a cada texto obtido.
* obter uma amostra dos dados para posterior validação manual da qualidade dos textos.

A estratégia foi desenhada para capturar dados intencionalmente desbalanceados e com ruídos bi-idioma, simulando um cenário real de engenharia de dados.

## Coleta e Data Labeling

#### Quantidade de Dados

Para realizar um MVP iremos coletar cerca de 1800 registros, sendo 300 de cada categoria. 
Considerando que a maior parte dos materiais de temas que envolvem tecnologia serem disponibilizados na lingua inglesa, o modelo será treinado nessa lingua. 

#### Rotulagem

Para realizar a rotulagem dos textos, foram definidas 6 categorias e, para cada uma delas, 5 subcategorias: 

⚠️ Descrição pendente
* incluir descrição das catergoris e subcategorias>


Cada subcategoria será utilizada como parametro (tag) no consumo da API. Os textos retornados serão associados a categoria que representa a subcategoria, gerando assim o target.



In [1]:
# Importando bibliotecas
import requests
import time
import pandas as pd

from IPython.display import display
pd.reset_option('display.max_colwidth')

In [2]:
# Estrutura Hierárquica: 6 Categorias Principais, cada uma com 6 Subtags
mapeamento_categorias = {
    'Frontend': ['frontend','javascript','react','css','typescript','nextjs'],
    'Backend' : ['backend','springboot','java','api','php','csharp'],
    'Cloud'   : ['cloud','kubernetes','oci','aws','azure','docker'],
    'Database': ['database','postgres','nosql','sql','mongodb','crud'],
    'Security': ['security','mfa','cybersecurity','infosec','cryptography','ethicalhacking'],
    'DataScience': ['datascience','machinelearning','pandas','ai','python','deeplearning']
}

In [3]:
# Buscar artigos por categoria
lista_artigos = []

# Buscar artigos por tags e associar target (categorias)
for categoria, subcategorias in mapeamento_categorias.items():
    print(f"\n📁 Coletando textos para a categoria {categoria}")
    
    for subcategoria in subcategorias:
        print(f"  └─ 🏷️ Buscando por tag: {subcategoria}")
        
       # Para cada subcategoria, coleta 60 artigos
        try:
            artigos = requests.get(
                        "https://dev.to/api/articles",
                        params={
                            "tag": subcategoria,
                            "per_page": 60
                        },
                        timeout=20
            ).json()
        except requests.exceptions.RequestException as e:
            print(f"Erro ao buscar tag '{subcategoria}': {e}")
            continue
        
        # Para cada artigo, buscar o texto completo por id
        for artigo in artigos:
            try:  
                detalhe = requests.get(
                    f"https://dev.to/api/articles/{artigo["id"]}",
                    timeout=20
                ).json()
            except requests.exceptions.RequestException as e:
                print(f"Erro na subcategoria '{subcategoria}' ao buscar artigo '{artigo["id"]}': {e}")
                continue

            # Gera lista completa de artigos
            lista_artigos.append({
                "categoria": categoria,
                "subcategoria": subcategoria,
                "texto": detalhe.get("body_markdown", ""),
                **artigo              
            })
                        
            # Pausa entre requisições para não estressar a API
            time.sleep(0.5)

# Gera dataframe bruto            
df_artigos = pd.DataFrame(lista_artigos)   


📁 Coletando textos para a categoria Frontend
  └─ 🏷️ Buscando por tag: frontend
  └─ 🏷️ Buscando por tag: javascript
  └─ 🏷️ Buscando por tag: react
  └─ 🏷️ Buscando por tag: css
  └─ 🏷️ Buscando por tag: typescript
  └─ 🏷️ Buscando por tag: nextjs

📁 Coletando textos para a categoria Backend
  └─ 🏷️ Buscando por tag: backend
  └─ 🏷️ Buscando por tag: springboot
  └─ 🏷️ Buscando por tag: java
  └─ 🏷️ Buscando por tag: api
  └─ 🏷️ Buscando por tag: php
  └─ 🏷️ Buscando por tag: csharp

📁 Coletando textos para a categoria Cloud
  └─ 🏷️ Buscando por tag: cloud
  └─ 🏷️ Buscando por tag: kubernetes
  └─ 🏷️ Buscando por tag: oci
  └─ 🏷️ Buscando por tag: aws
  └─ 🏷️ Buscando por tag: azure
  └─ 🏷️ Buscando por tag: docker

📁 Coletando textos para a categoria Database
  └─ 🏷️ Buscando por tag: database
  └─ 🏷️ Buscando por tag: postgres
  └─ 🏷️ Buscando por tag: nosql
  └─ 🏷️ Buscando por tag: sql
  └─ 🏷️ Buscando por tag: mongodb
  └─ 🏷️ Buscando por tag: crud

📁 Coletando textos para a cat

## Análise Exploratória

In [4]:
df_artigos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2160 entries, 0 to 2159
Data columns (total 32 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   categoria                 2160 non-null   object 
 1   subcategoria              2160 non-null   object 
 2   texto                     2160 non-null   object 
 3   type_of                   2160 non-null   object 
 4   id                        2160 non-null   int64  
 5   title                     2160 non-null   object 
 6   description               2160 non-null   object 
 7   readable_publish_date     2160 non-null   object 
 8   slug                      2160 non-null   object 
 9   path                      2160 non-null   object 
 10  url                       2160 non-null   object 
 11  comments_count            2160 non-null   int64  
 12  public_reactions_count    2160 non-null   int64  
 13  collection_id             236 non-null    float64
 14  publishe

Podemos verificar que:
* O retorno foi de 2160 artigos. Não há dados faltantes na coluna "texto" e na coluna "tags"
* Há diversas colunas que podem possibilitar a identificação dos autores dos artigos. Todas serão ocultadas.

In [5]:
# Definindo colunas que serão ocultadas
# Motivo: evitar identificação do autor do texto
colunas_ocultar = ['path', 'url','canonical_url',"slug", "social_image","cover_image", 'user',"organization"]

In [6]:
# Conhecendo os dados 
df_artigos.drop(columns=colunas_ocultar).head(2)

,categoria,subcategoria,texto,type_of,id,title,description,readable_publish_date,comments_count,public_reactions_count,...,positive_reactions_count,created_at,edited_at,crossposted_at,published_at,last_comment_at,reading_time_minutes,tag_list,tags,flare_tag
0,Frontend,frontend,Choosing a state management library feels a bi...,article,4175536,Stop Over Engineering Your Frontend State: A D...,Choosing a state management library feels a bi...,Jul 18,0,0,...,0,2026-07-18T19:29:08Z,2026-07-18T19:31:43Z,None,2026-07-18T19:29:08Z,2026-07-18T19:29:08Z,3,"[architecture, frontend, react, typescript]","architecture, frontend, react, typescript",NaN
1,Frontend,frontend,> [_Artikel ini juga tersedia dalam bahasa Ing...,article,4172624,Kenapa Banyak Frontend Developer Memakai Next....,Artikel ini juga tersedia dalam bahasa Inggris...,Jul 18,0,0,...,0,2026-07-18T09:15:16Z,2026-07-18T10:01:12Z,None,2026-07-18T09:25:24Z,2026-07-18T09:25:24Z,3,"[webdev, frontend, nextjs, vue]","webdev, frontend, nextjs, vue",NaN


In [ ]:
# Conferindo conteúdo do texto com o postado em site - via url 
# Código ocultado e output excluído para não expor os dados de identificação
'''
print(
    df_artigos.loc[
        df_artigos["id"] == 4172624,
        ["categoria", "subcategoria", "url", "social_image", "cover_image"]
    ].to_string()
)
'''

In [8]:
# Quantidade obtido por categoria 
df_artigos['categoria'].value_counts()

categoria
Frontend       360
Backend        360
Cloud          360
Database       360
Security       360
DataScience    360
Name: count, dtype: int64

In [9]:
# Quantidade por subcategoria
df_artigos["subcategoria"].value_counts()

subcategoria
frontend           60
javascript         60
react              60
css                60
typescript         60
nextjs             60
backend            60
springboot         60
java               60
api                60
php                60
csharp             60
cloud              60
kubernetes         60
oci                60
aws                60
azure              60
docker             60
database           60
postgres           60
nosql              60
sql                60
mongodb            60
crud               60
security           60
mfa                60
cybersecurity      60
infosec            60
cryptography       60
ethicalhacking     60
datascience        60
machinelearning    60
pandas             60
ai                 60
python             60
deeplearning       60
Name: count, dtype: int64

In [10]:
# Quantidade de registros por lingua 
df_artigos['language'].value_counts(dropna=False)

language
en         2010
None         44
pt           33
es           29
de            8
fr            7
zh            6
vi            5
id            4
tr            3
ro            2
hu            2
ko            1
th            1
ja            1
hi            1
no            1
zh-Latn       1
nl            1
Name: count, dtype: int64

A maior parte dos registros são da lingua inglesa

In [11]:
# Visualizar dados sem language
df_artigos[df_artigos["language"].isna()].drop(columns=colunas_ocultar)

,categoria,subcategoria,texto,type_of,id,title,description,readable_publish_date,comments_count,public_reactions_count,...,positive_reactions_count,created_at,edited_at,crossposted_at,published_at,last_comment_at,reading_time_minutes,tag_list,tags,flare_tag
36,Frontend,frontend,<h2>The Interaction Gap</h2>\n<p>When crafting...,article,4154379,Banish UI Lag: Custom Hook Interaction Feedback ⚡,Liquid syntax error: Variable '{{% raw %}' was...,Jul 16,0,1,...,1,2026-07-16T04:31:11Z,None,None,2026-07-16T04:31:11Z,2026-07-16T04:31:11Z,3,"[react, frontend, javascript, webdev]","react, frontend, javascript, webdev",NaN
176,Frontend,react,<h2>The Interaction Gap</h2>\n<p>When crafting...,article,4154379,Banish UI Lag: Custom Hook Interaction Feedback ⚡,Liquid syntax error: Variable '{{% raw %}' was...,Jul 16,0,1,...,1,2026-07-16T04:31:11Z,None,None,2026-07-16T04:31:11Z,2026-07-16T04:31:11Z,3,"[react, frontend, javascript, webdev]","react, frontend, javascript, webdev",NaN
355,Frontend,nextjs,{% embed https://dev.to/qingyuan_yang_11c04dc6...,article,4153763,Open-source AI Interview Platform for chat/voi...,Building a Self-Hosted Voice AI Interview Plat...,Jul 16,0,0,...,0,2026-07-16T02:06:02Z,None,None,2026-07-16T02:06:02Z,2026-07-16T02:06:02Z,1,"[ai, nextjs, opensource, showdev]","ai, nextjs, opensource, showdev","{'name': 'showdev', 'bg_color_hex': '#091b47',..."
1044,Cloud,docker,{% embed https://dev.to/ladipo_samuel_7cfaa827...,article,4157032,dockerizing your fastapi application,Docker Series Part 3: Dockerizing Your First A...,Jul 16,0,0,...,0,2026-07-16T09:30:20Z,None,None,2026-07-16T09:30:20Z,2026-07-16T09:30:20Z,1,"[api, docker, python, tutorial]","api, docker, python, tutorial",NaN
1410,Database,crud,"আপনি যদি রিলেশনাল ডাটাবেজ নিয়ে কাজ করেন, তাহলে...",article,1786732,PostgreSQL পাওয়ার আপ: কুয়েরি নিয়ে আলোচনা,"আপনি যদি রিলেশনাল ডাটাবেজ নিয়ে কাজ করেন, তাহলে...",Mar 11 '24,0,0,...,0,2024-03-11T10:05:57Z,None,None,2024-03-11T10:05:56Z,2024-03-11T10:05:56Z,1,"[postgressql, query, crud, bangla]","postgressql, query, crud, bangla",NaN
1429,Database,crud,![](https://cdn-images-1.medium.com/max/1600/1...,article,1515738,Firebase Functions to Build a Serverless CRUD App,Firebase is a powerful backend service provide...,Jun 24 '23,0,0,...,0,2023-06-24T23:28:48Z,None,None,2023-06-24T23:28:47Z,2023-06-24T23:28:47Z,7,"[webdev, javascript, firebase, crud]","webdev, javascript, firebase, crud",NaN
1430,Database,crud,## Introduction\nThis is a simple guide meant ...,article,1471539,"Basic CRUD with Prisma, Fastify and Node",Introduction This is a simple guide meant to...,May 30 '23,0,13,...,13,2023-05-17T20:29:19Z,None,None,2023-05-30T19:09:54Z,2023-05-30T19:09:54Z,4,"[beginners, fastify, crud, prisma]","beginners, fastify, crud, prisma",NaN
1431,Database,crud,Cypher is a query language specifically design...,article,1485465,Beginner's Guide to CRUD Operations in Apache age,Cypher is a query language specifically design...,May 30 '23,0,2,...,2,2023-05-30T08:22:22Z,None,None,2023-05-30T08:22:21Z,2023-05-30T08:22:21Z,2,"[apacheage, database, crud, cyphe]","apacheage, database, crud, cyphe",NaN
1432,Database,crud,\n![Image description](https://dev-to-uploads....,article,1439646,Building a Simple CRUD Contract with Solidity ...,"If you're new to Solidity, the programming lan...",Apr 18 '23,0,0,...,0,2023-04-18T10:44:47Z,None,None,2023-04-18T10:44:47Z,2023-04-18T10:44:47Z,3,"[solidity, crud]","solidity, crud",NaN
1434,Database,crud,I have an interview coming up and needed to qu...,article,1469391,Part 1: Setting up postgres for a simple C.R.U...,I have an interview coming up and needed to qu...,May 16 '23,0,3,...,3,2023-05-16T01:55:13Z,None,None,2023-05-16T01:55:13Z,2023-05-16T01:55:13Z,3,"[node, express, postgressql, crud]","node, express, postgressql, crud",NaN


Registros com outros idiomas e sem language podem ser identificados e desconsiderados, pois modelo será treinado em inglês 

In [12]:
# Gerar csv desconsiderando colunas de identificação
colunas = df_artigos.columns.difference(colunas_ocultar)

df_artigos.to_csv(
    "artigos.csv",
    columns=colunas,
    index=False,
    encoding="utf-8-sig"
)

#### Filtro - Textos em Inglês

In [13]:
# Filtro dos textos em inglês 
df_artigos_en = df_artigos[df_artigos["language"] == "en"]

In [14]:
# Verificando quantidade de linhas e colunas restantes 
df_artigos_en.drop(columns=colunas_ocultar).shape

(2010, 24)

In [15]:
# Quantidade restante por subcategoria 
df_artigos_en['subcategoria'].value_counts()

subcategoria
cryptography       60
infosec            60
security           60
javascript         59
ethicalhacking     59
java               59
azure              59
api                59
database           59
aws                59
nosql              58
cybersecurity      58
python             58
kubernetes         58
datascience        58
ai                 58
oci                58
cloud              58
typescript         58
nextjs             58
csharp             56
postgres           56
react              56
backend            56
css                55
frontend           55
springboot         54
php                54
mongodb            54
docker             53
sql                52
machinelearning    52
mfa                49
pandas             47
crud               45
deeplearning       43
Name: count, dtype: int64

In [16]:
# Conferindo a existencia do registro utilizado para teste antes de filtro para language = ingles
# O registro apresenta lingua diferente do inglês 
registro = df_artigos[df_artigos["title"].str.contains("Python数", na=False)]
registro

,categoria,subcategoria,texto,type_of,id,title,description,readable_publish_date,slug,path,...,edited_at,crossposted_at,published_at,last_comment_at,reading_time_minutes,tag_list,tags,user,organization,flare_tag
1940,DataScience,pandas,如果你想用Python做数据分析，**Pandas**是你必须掌握的第一个工具。它是数据科学...,article,3690273,Python数据科学入门：从零开始掌握Pandas,如果你想用Python做数据分析，Pandas是你必须掌握的第一个工具。它是数据科学家手中的...,May 18,pythonshu-ju-ke-xue-ru-men-cong-ling-kai-shi-z...,/wdsega/pythonshu-ju-ke-xue-ru-men-cong-ling-k...,...,None,None,2026-05-18T01:36:03Z,2026-05-18T01:36:03Z,1,"[python, datascience, pandas, tutorial]","python, datascience, pandas, tutorial","{'name': 'WDSEGA', 'username': 'wdsega', 'twit...",NaN,NaN


In [17]:
# Conferindo inexistência do registro após filtro de texto em inglês
registro = df_artigos_en[df_artigos_en["title"].str.contains("Python数", na=False)]
registro

,categoria,subcategoria,texto,type_of,id,title,description,readable_publish_date,slug,path,...,edited_at,crossposted_at,published_at,last_comment_at,reading_time_minutes,tag_list,tags,user,organization,flare_tag


In [18]:
# Gerar csv somente com textos em inglês e desconsiderando colunas de identificação
df_artigos_en.to_csv(
    "artigos_en.csv",
    columns=colunas,
    index=False,
    encoding="utf-8-sig"
)

## Amostra

A partir dos dados coletados será obtida uma amostra com 10% dos dados. A amostra terá textos das 6 categorias definidas e de todas as subcategorias associadas.

O objetivo final é validar, de forma manual, todos os textos da amostra afim de conferir se os mesmos são realmente da categoria a que foi associado. Dessa forma iremos validar a qualidade dos dados e consequentemente viabilizar a utilização dos mesmos no Hackathon ONE - Projeto TechMind. 

In [66]:
df_artigos_en_amostra = pd.DataFrame()

In [67]:
# Gerar amostra considerando 10% de cada categoria
df_artigos_en_amostra = (
    df_artigos_en
    .groupby(["categoria", "subcategoria"], group_keys=False)
    .apply(lambda x: x.sample(frac=0.1, random_state=42))
)

C:\Users\Patricia\AppData\Local\Temp\ipykernel_24524\727679874.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=0.1, random_state=42))


In [68]:
# Verificando estrutura da amostra
df_artigos_en_amostra.info()

<class 'pandas.core.frame.DataFrame'>
Index: 204 entries, 540 to 1494
Data columns (total 32 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   categoria                 204 non-null    object 
 1   subcategoria              204 non-null    object 
 2   texto                     204 non-null    object 
 3   type_of                   204 non-null    object 
 4   id                        204 non-null    int64  
 5   title                     204 non-null    object 
 6   description               204 non-null    object 
 7   readable_publish_date     204 non-null    object 
 8   slug                      204 non-null    object 
 9   path                      204 non-null    object 
 10  url                       204 non-null    object 
 11  comments_count            204 non-null    int64  
 12  public_reactions_count    204 non-null    int64  
 13  collection_id             22 non-null     float64
 14  published_ti

In [69]:
# Distribuição da amostra
df_artigos_en_amostra.groupby(["categoria", "subcategoria"]).size()

categoria    subcategoria   
Backend      api                6
             backend            6
             csharp             6
             java               6
             php                5
             springboot         5
Cloud        aws                6
             azure              6
             cloud              6
             docker             5
             kubernetes         6
             oci                6
DataScience  ai                 6
             datascience        6
             deeplearning       4
             machinelearning    5
             pandas             5
             python             6
Database     crud               4
             database           6
             mongodb            5
             nosql              6
             postgres           6
             sql                5
Frontend     css                6
             frontend           6
             javascript         6
             nextjs             6
             react 

São 204 registros na amostra. A maior parte das subcategorias estão representadas com 5 ou 6 artigos. As subcategorias crud e deeplearning estão sendo representadas por 4 artigos             

In [70]:
# Verificando duplicidade na amostra (Duplicidade = id + categoria iguais)
df_artigos_en_amostra[
    df_artigos_en_amostra.duplicated(subset=["id", "categoria"], keep=False)
][["id", "categoria", "subcategoria", "title", "texto", "tags"]].sort_values(["id", "categoria"])

,id,categoria,subcategoria,title,texto,tags
1980,4167079,DataScience,ai,Porting a 128-expert MoE (Gemma-4 26B-A4B) to ...,"---\ntitle: ""Porting a 128-expert MoE (Gemma-4...","machinelearning, aws, ai, python"
2040,4167079,DataScience,python,Porting a 128-expert MoE (Gemma-4 26B-A4B) to ...,"---\ntitle: ""Porting a 128-expert MoE (Gemma-4...","machinelearning, aws, ai, python"
594,4168837,Backend,api,"A Practical Guide to API Types: REST, GraphQL,...",\n## A Practical Guide to API Types and When t...,"api, webdev, backend, beginners"
393,4168837,Backend,backend,"A Practical Guide to API Types: REST, GraphQL,...",\n## A Practical Guide to API Types and When t...,"api, webdev, backend, beginners"
300,4177014,Frontend,nextjs,GPT-5.6 Just Closed a 30-Year Math Gap with a ...,"Alright, so I'm scrolling Reddit the other day...","ai, programming, typescript, nextjs"
240,4177014,Frontend,typescript,GPT-5.6 Just Closed a 30-Year Math Gap with a ...,"Alright, so I'm scrolling Reddit the other day...","ai, programming, typescript, nextjs"


Há registros duplicados na amostra, ou seja, possuem o mesmo id e mesma categoria, porém a subcategoria é diferente. Isso ocorre porque um mesmo artigo pode ter até 4 tagas associadas a ele. A validação manual irá avaliar também as subcategorias representadas na coluna tags, portando não faz sentido manter esses registros duplicados.
 

In [73]:
# Removendo registros duplicados da amostra
df_artigos_en_amostra = df_artigos_en_amostra.drop_duplicates(subset=["id", "categoria"])

In [74]:
# Quantidade de artigos da amostra
df_artigos_en_amostra.shape

(201, 32)

In [75]:
# Verificando os id´s unicos
df_artigos_en_amostra["id"].nunique()

200

In [76]:
# verificando id´s duplicados em categorias diferentes
df_artigos_en_amostra[df_artigos_en_amostra.duplicated(subset="id", keep=False)].sort_values("id")

,categoria,subcategoria,texto,type_of,id,title,description,readable_publish_date,slug,path,...,edited_at,crossposted_at,published_at,last_comment_at,reading_time_minutes,tag_list,tags,user,organization,flare_tag
1993,DataScience,ai,"Alright, so I'm scrolling Reddit the other day...",article,4177014,GPT-5.6 Just Closed a 30-Year Math Gap with a ...,"Alright, so I'm scrolling Reddit the other day...",Jul 19,gpt-56-just-closed-a-30-year-math-gap-with-a-p...,/hanzla/gpt-56-just-closed-a-30-year-math-gap-...,...,None,None,2026-07-19T04:00:10Z,2026-07-19T04:00:10Z,3,"[ai, programming, typescript, nextjs]","ai, programming, typescript, nextjs","{'name': 'Hanzla Baig', 'username': 'hanzla', ...",NaN,NaN
300,Frontend,nextjs,"Alright, so I'm scrolling Reddit the other day...",article,4177014,GPT-5.6 Just Closed a 30-Year Math Gap with a ...,"Alright, so I'm scrolling Reddit the other day...",Jul 19,gpt-56-just-closed-a-30-year-math-gap-with-a-p...,/hanzla/gpt-56-just-closed-a-30-year-math-gap-...,...,None,None,2026-07-19T04:00:10Z,2026-07-19T04:00:10Z,3,"[ai, programming, typescript, nextjs]","ai, programming, typescript, nextjs","{'name': 'Hanzla Baig', 'username': 'hanzla', ...",NaN,NaN


Considerando o total de 201 registros na amostra, há 1 artigo com o mesmo id mas em categorias diferentes. Nesse caso vamos manter na amostra para poder validar manualmente e analisar melhor a situação.

In [87]:
# Gerar csv da amostra
df_artigos_en_amostra[["id", "categoria", "subcategoria", "title", "texto", "tags"]].to_csv(
    "artigos_en_amostra.csv", index=False
)

## Validação da Amostra


A amostra gerada será validada manualmente: cada artigo será lido e avaliado quanto à sua coerência com a categoria à qual foi associado, bem como quanto à coerência com as tags retornadas pela API e que são utilizadas obter os artigos.

O time se dividirá para realizar essa validação.

O arquivo CSV será importado no Google Sheets e compartilhado via Drive, onde serão incluídas colunas adicionais para que cada membro da equipe documente sua avaliação. Posteriormente, os resultados serão consolidados para uma análise geral da amostra.

## Primeiros Insights

A execução bem-sucedida dos scripts de coleta e amostragem resultou na estruturação e armazenamento dos dados em três arquivos CSV distintos, garantindo a organização do projeto:

 1. `artigos.csv` (Base Completa): Contém 2.160 artigos obtidos pela API. Foram obtidos 60 artigos por subcategoria, sendo 360 para cada categoria. Este arquivo preserva os dados originais. Foram desconsideradas as colunas que poderiam identificar o(s) autor(es).

 2. `artigos_en.csv` (Base em Inglês): Contém 2.010 artigos em inglês. Esses artigos foram obtidos da base completa (artigos.csv) realizando um filtro através da coluna language. Este arquivo preserva os dados originais e será a base oficial de treinamento do modelo. 

 3. `artigos_en_amostra.csv` (Base de Auditoria): Contém 201 artigos sorteados aleatoriamente (cerca de 10% por subcategoria) a partir da base em inglês (artigos_en.csv). Esse arquivo será utilizado para realizar a validação manual dos textos, afim de viabilizar a utilização da base completa no projeto.

Até o momento, a partir da análise dessas duas bases e da inspeção visual humana realizada na amostra, foram gerados os seguintes insights:

* **Aderência dos Temas:** A checagem humana confirmou que o conteúdo dos artigos trata diversos temas da área de tecnologia, o que está alinhado com o objetivo do projeto.

* **Múltiplas Tags:** Os autores utilizam várias tags por post (ex: um artigo de `#react` também traz `#frontend` e `#javascript`), provando que os dados capturaram o ecossistema real e interligado da tecnologia.

* **Presença de Elementos de Programação:** Os textos extraídos vêm acompanhados de blocos de códigos reais de programação (sintaxes de software), links e marcações visuais de Markdown.

## Conclusão e Recomendações

Com base nos insights obtidos na fase de obtenção de dados, conclui-se que o banco de dados é rico e provavelmente viável, mas exige cuidados antes de alimentar a inteligência artificial.

Para preparar os textos para o treinar o modelo, recomenda-se as seguintes ações:

* **Remoção de Códigos de Programação:** Criar um script de limpeza para deletar os exemplos de códigos colados pelos autores, evitando que termos como const, return ou chaves {} confundam o classificador.

* **Limpeza de Formatação:** Aplicar filtros para apagar links, imagens e caracteres especiais do Markdown que atuam como ruído.

* **Normalização Textual:** Converter todas as palavras para minúsculas e aplicar técnicas de Processamento de Linguagem Natural (PLN) para preparar o texto final.

## Referências



* <a id="ref1"></a>**[1]** DEV COMMUNITY. *Site*. Disponível em: <https://dev.to/>. Acesso em: 9 jul. 2026.

* <a id="ref2"></a>**[2]** FOREM. *Sobre dev.to*. Disponível em: <https://github.com/forem/forem/>. Acesso em: 9 jul. 2026.

* <a id="ref3"></a>**[3]** API. *Versões da API*. Disponível em: <https://developers.forem.com/api>. Acesso em: 9 jul. 2026.

* <a id="ref4"></a>**[4]** Forem API V1. *Documentação da API*. Disponível em: <https://developers.forem.com/api/v1>. Acesso em: 9 jul. 2026.